In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pickle
from pathlib import Path
import pandas as pd
from transformers import AutoTokenizer
from transformers import BertModel
import sys
import torch

sys.path.append(str(Path("..")))

from utils.ai_human_predictor import AiHumanPredictor

device = "cuda"

In [3]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
encoder = BertModel.from_pretrained("bert-base-uncased")

predictor = AiHumanPredictor(tokenizer, encoder, "../models/sgd_classifier.pkl", device)
predictor.load_head()

Head model loaded from ../models/sgd_classifier.pkl successfully


In [4]:
model_dir = "models"


model_files = [file for file in Path.iterdir(Path(f"../{model_dir}")) if not Path.is_dir(file)]

model_dict = {}
for file in model_files:
    with open(file, "rb") as f:
        model_dict[file] = pickle.load(f)
model_dict

{PosixPath('../models/linear_svc.pkl'): LinearSVC(random_state=42),
 PosixPath('../models/sgd_classifier.pkl'): SGDClassifier(random_state=42),
 PosixPath('../models/random_forest.pkl'): RandomForestClassifier(random_state=42),
 PosixPath('../models/logistic_regression.pkl'): LogisticRegressionCV(random_state=42),
 PosixPath('../models/decision_tree.pkl'): DecisionTreeClassifier(random_state=42)}

In [6]:
def get_feature_importances(model):
    if hasattr(model, "coef_"):
        return model.coef_.ravel()
    elif hasattr(model, "feature_importances_"):
        return model.feature_importances_


df = (
    pd.DataFrame(
        {
            model_path.stem: get_feature_importances(model)
            for model_path, model in model_dict.items()
        }
    )
    .reset_index(drop=False)
    .rename(columns={"index": "feature_num"})
)
df.head()

,feature_num,linear_svc,sgd_classifier,random_forest,logistic_regression,decision_tree
0,0,-0.137768,0.309068,0.000692,-0.040862,0.000329
1,1,0.132761,0.761442,0.000527,0.657918,0.000317
2,2,-0.124179,-1.096989,0.027710,-0.490668,0.016137
3,3,0.338633,0.778045,0.001486,0.950695,0.001738
4,4,-0.188860,-0.608605,0.000789,-0.574058,0.000887


In [7]:
df.drop(columns="feature_num").corr(method="spearman")

,linear_svc,sgd_classifier,random_forest,logistic_regression,decision_tree
linear_svc,1.000000,0.866857,0.010242,0.974079,0.049446
sgd_classifier,0.866857,1.000000,0.002148,0.935238,0.040042
random_forest,0.010242,0.002148,1.000000,0.000140,0.630549
logistic_regression,0.974079,0.935238,0.000140,1.000000,0.044082
decision_tree,0.049446,0.040042,0.630549,0.044082,1.000000


In [30]:
sample = "this is a human made piece of text. I have written it"
sample2 = """I’m trying to fetch some data from an API using async/await, but instead of getting the JSON object back, my function keeps returning a Promise.
Here’s my code:"""

print(predictor.forward(sample))
predictor.visualize_token_importance(sample)

[0]


In [31]:
predictor.get_token_importance_by_gradients(sample)[1].mean()

np.float32(2.0429165e-06)